# Question 17 - 13. Roman to Integer

Roman numerals are represented by seven different symbols: `I`, `V`, `X`, `L`, `C`, `D` and `M`.

| Symbol | Value |
|---|---|
| I | 1 |
| V | 5 |
| X | 10 |
| L | 50 |
| C | 100 |
| D | 500 |
| M | 1000 |

For example, `2` is written as `II` in Roman numeral, just two ones added together. `12` is written as `XII`, which is simply `X + II`. The number `27` is written as `XXVII`, which is `XX + V + II`.

Roman numerals are usually written largest to smallest from left to right. However, the numeral for four is not `IIII`. Instead, the number four is written as `IV`. Because the one is before the five we subtract it making four. The same principle applies to the number nine, which is written as `IX`. There are six instances where subtraction is used:

- `I` can be placed before `V` (5) and `X` (10) to make 4 and 9.
- `X` can be placed before `L` (50) and `C` (100) to make 40 and 90.
- `C` can be placed before `D` (500) and `M` (1000) to make 400 and 900.

Given a roman numeral, convert it to an integer.

**Example 1:**

    Input: s = "III"
    Output: 3
    Explanation: III = 3.

**Example 2:**

    Input: s = "LVIII"
    Output: 58
    Explanation: L = 50, V = 5, III = 3.

**Example 3:**

    Input: s = "MCMXCIV"
    Output: 1994
    Explanation: M = 1000, CM = 900, XC = 90 and IV = 4.

**Constraints:**

- `1 <= s.length <= 15`
- `s` contains only the characters `('I', 'V', 'X', 'L', 'C', 'D', 'M')`.
- It is **guaranteed** that `s` is a valid roman numeral in the range `[1, 3999]`.

---
## 🎯 Interview Walkthrough: Brute Force → Optimized

**Pattern:** Lookup table (dict) + one scan: subtract a symbol when a bigger one follows it

📘 **Revise first:** [Pattern H · simulate carefully](./0-concepts-array-and-string.ipynb) · [Part 6 · dict lookups](./0-concepts-array-and-string.ipynb) in the concepts notebook

| Approach | Time | Space |
|---|---|---|
| 1️⃣ Brute force (rewrite the 6 special pairs, then add) | O(n) | O(n) |
| 2️⃣ Optimized (compare each symbol with the next) | O(n) | O(1) |

> 🧠 = plain-English intuition (how you'd explain it to a friend) · 🗣️ = **interview script**: what to say out loud at each step.

### 🧠 The problem in plain English

Roman numerals are letters with values: `I = 1, V = 5, X = 10, L = 50, C = 100, D = 500, M = 1000`.

**Usually you just add them up, biggest first:** `LVIII` = 50 + 5 + 1 + 1 + 1 = **58**.

**The twist:** to avoid writing four of the same letter, Romans put a **smaller letter in front of a bigger one** to mean "subtract":
`IV` = 5 − 1 = **4**, `IX` = 9, `XL` = 40, `XC` = 90, `CD` = 400, `CM` = 900.

**Everyday picture: reading a price tag with a "minus" hint.** Read left to right. For each letter, peek at the **next** letter:
- If the next letter is **bigger**, this letter is a "minus" → subtract it.
- Otherwise → add it.

`MCMXCIV`: M (+1000), C before M (−100), M (+1000), X before C (−10), C (+100), I before V (−1), V (+5) → **1994**.

That single "peek at the next letter" rule covers all six special cases, so there's no need to memorise them.

### Step 0 · Clarify before coding

🗣️ *"Normally the symbols go from largest to smallest and I add them. The exception is the six subtractive pairs, where a smaller symbol comes before a larger one. The input is always a valid numeral between 1 and 3999."*

| Question to ask | Answer | Why it matters |
|---|---|---|
| Always valid? | **Yes.** | No need to reject things like `IIII` or `IC`. |
| Uppercase only? | **Yes.** | |
| Length? | At most 15. | Speed isn't the issue; **clear, correct logic** is. |

### Step 1 · Brute force: the "obvious" way

**Plain-English idea:** list the six special pairs explicitly. Rewrite each one into plain addition (`IV` → `IIII`, `CM` → `DCCCC`, …), then add up every letter.

🗣️ *"A straightforward approach is to hard-code the six subtractive pairs, replace them with their additive equivalents, and then just sum the letter values. It works, but it relies on a table of special cases and builds several new strings."*

**What's not great:** speed is fine (O(n)), but it's **six special cases** you must remember and type correctly, and each `replace` builds a new string (extra memory).

In [1]:
class SolutionBrute:
    VALUES = {"I": 1, "V": 5, "X": 10, "L": 50, "C": 100, "D": 500, "M": 1000}
    EXPAND = {"IV": "IIII", "IX": "VIIII", "XL": "XXXX", "XC": "LXXXX", "CD": "CCCC", "CM": "DCCCC"}

    def romanToInt(self, s: str) -> int:
        for pair, additive in self.EXPAND.items():
            s = s.replace(pair, additive)          # remove the "minus" cases
        return sum(self.VALUES[ch] for ch in s)    # now it's pure addition

### Step 2 · Optimize: the "aha" moment

> 💡 **Aha:** all six special cases follow **one rule**. In a valid numeral, a symbol is subtracted **exactly when it's smaller than the symbol right after it**. So instead of listing pairs, just compare each symbol with its neighbour.

**What data structure?** A **dictionary** (`dict`) mapping each letter to its value, so `VALUES["M"]` gives 1000 instantly (O(1)). There are only 7 entries.

**Algorithm:** walk the string once. For each letter, look up its value and the next letter's value. Smaller than the next → subtract; otherwise → add.

🗣️ *"Rather than hard-coding the six pairs, I'll use the rule that produces them: if a symbol's value is less than the next symbol's value, subtract it, otherwise add it. I store the seven symbol values in a dictionary for constant-time lookup and make one pass, O(n) time, O(1) space."*

#### 🤔 Why this approach, and why not the others?

| Option | Verdict | Plain-English reason |
|---|---|---|
| Hard-code pairs + `replace` | ⚠️ | Works, but six special cases, extra strings, easy to typo. |
| Long `if/elif` chain for each letter | ⚠️ | Verbose. A lookup table is shorter and less error-prone. |
| **dict lookup + compare with the next letter** | ✅ **best** | One general rule, one pass, constant memory. |
| Scan **right to left**, subtract when smaller than the last value seen | ✅ equivalent | Same cost. Some find it more natural; either is fine. |
| Regular expressions | ❌ | Overkill for 15 characters and one simple rule. |

In [2]:
class Solution:
    VALUES = {"I": 1, "V": 5, "X": 10, "L": 50, "C": 100, "D": 500, "M": 1000}

    def romanToInt(self, s: str) -> int:
        total = 0
        for i, ch in enumerate(s):
            value = self.VALUES[ch]
            next_value = self.VALUES[s[i + 1]] if i + 1 < len(s) else 0
            if value < next_value:
                total -= value      # a smaller symbol before a bigger one means "minus"
            else:
                total += value
        return total

### Step 3 · Toy example walkthrough (tell it like a story)

`s = "MCMXCIV"`

| i | letter | value | next value | smaller than next? | action | running total |
|---|---|---|---|---|---|---|
| 0 | M | 1000 | C = 100 | ❌ | +1000 | 1000 |
| 1 | C | 100 | M = 1000 | ✅ | **−100** | 900 |
| 2 | M | 1000 | X = 10 | ❌ | +1000 | 1900 |
| 3 | X | 10 | C = 100 | ✅ | **−10** | 1890 |
| 4 | C | 100 | I = 1 | ❌ | +100 | 1990 |
| 5 | I | 1 | V = 5 | ✅ | **−1** | 1989 |
| 6 | V | 5 | (none) | ❌ | +5 | **1994** ✅ |

**Narrated:** "M is a thousand. C comes before a bigger M, so it's minus 100. Then M, another thousand. X before a bigger C, minus 10. C, plus 100. I before a bigger V, minus 1. V, plus 5. That's 1994. The three minuses are exactly CM, XC and IV, and I never had to list them."

In [3]:
cases = [
    ("III", 3), ("LVIII", 58), ("MCMXCIV", 1994),
    ("IV", 4), ("IX", 9), ("XL", 40), ("CD", 400),
    ("MMMCMXCIX", 3999), ("I", 1),
]
for s, expected in cases:
    assert SolutionBrute().romanToInt(s) == expected
    assert Solution().romanToInt(s) == expected
    print(s, "->", expected)

# check every value 1..3999 using a reference encoder
def to_roman(num):
    table = [(1000, "M"), (900, "CM"), (500, "D"), (400, "CD"), (100, "C"), (90, "XC"),
             (50, "L"), (40, "XL"), (10, "X"), (9, "IX"), (5, "V"), (4, "IV"), (1, "I")]
    out = []
    for v, sym in table:
        q, num = divmod(num, v)
        out.append(sym * q)
    return "".join(out)

for n in range(1, 4000):
    assert Solution().romanToInt(to_roman(n)) == n
print("All tests passed ✅ (including every value 1..3999)")

III -> 3
LVIII -> 58
MCMXCIV -> 1994
IV -> 4
IX -> 9
XL -> 40
CD -> 400
MMMCMXCIX -> 3999
I -> 1
All tests passed ✅ (including every value 1..3999)


### Step 4 · Complexity in plain words, and wrap up

| | Time | Extra memory | In plain words |
|---|---|---|---|
| Rewrite pairs + sum | O(n) | O(n) | Six string rewrites, each making a new string. |
| **Compare with next** | **O(n)** | **O(1)** | One pass, a 7-entry table that never grows. |

**Why O(n) time?** One loop over the letters, with two dictionary lookups and a comparison each. (With n ≤ 15 this is tiny, but still say the general complexity.)

**Why O(1) memory?** The dictionary always has exactly 7 entries, whatever the input.

**Edge cases to mention:** a single letter, the largest value `MMMCMXCIX` (3999), each of the six subtractive pairs.

**Likely follow-ups:**
- *Validate the input too?* → Needs extra rules (at most 3 repeats; only I, X, C can be subtracted, and only from the next two bigger symbols). The compare-with-next rule alone would accept invalid strings like `IC`.
- *The reverse direction?* → Integer to Roman (Q18).

---

#### 🗣️ Full interview script (about 45 seconds)

*"Roman numerals are normally added from largest to smallest, except for six subtractive pairs like IV and CM.*

*I could hard-code those six pairs, but they all follow one rule: a symbol is subtracted exactly when it's smaller than the symbol after it. So I keep a dictionary of the seven symbol values, walk the string once, and for each symbol either subtract it, if the next symbol is bigger, or add it.*

*That's O(n) time with a constant-size table, so O(1) space."*